# Modelling Debt Risk on Different Income Levels Using Econophysics — Simulator v2

**Open-bath bipartite kinetic exchange model** of Malaysian household debt risk (B40 / M40 / T20) under four macro shocks: an OPR hike, RON95 subsidy removal, an inflation spike, and all three combined.

**How to use:** `Runtime → Run all`. With the default settings (30,000 households, 20 seeds) it takes about 2–3 minutes. Set `FULL_SIZE = True` in the Monte Carlo cell for the 120,000-household version in the research note.

### What changed from v1 (the original notebook)
| Issue in v1 | Fix in v2 |
|---|---|
| Households never received wages, so ~96% ended in debt | Each household is paid weekly by its employer firm |
| OPR changes had no effect (3% / 4.5% / 8% gave the same result) | Loans carry a balance; floating loans are re-priced at OPR + spread every week |
| Higher inflation *reduced* debt | Money flows are closed: firms earn from spending, pay wages, and pay profit out to the bath |
| Every class got the same loans, so "B40 defaults most" was built in | Loan take-up and size depend on income class |
| K-S validation compared the model with itself | Calibration checked against published DOSM figures; K-S helper ready for real data |
| One scenario, one run | Five scenarios × many seeds, with common random numbers and 5–95% bands |

> ⚠️ **Placeholder parameters.** Income is calibrated to DOSM HIES 2022. Saving propensities, loan take-up, loan sizes, living-cost shares and fuel shares are **illustrative placeholders** — replace them with BNM / AKPK / DOSM data (see the last section) before quoting the numbers in your thesis.

## 1. Model engine
All model code is in this one cell. Each weekly step: wages in → external demand → essential spending (CPI- and fuel-indexed) → loan servicing → card interest on negative balances (Eq. 3) → kinetic exchange (Eq. 1–2) → profit payout → distress checks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# FYP: Modelling Debt Risk on Different Income Levels Using Econophysics
# Open-Bath Bipartite Kinetic Exchange Model -- v2 (fixed engine)
#
# Changes from v1 (cell 5 of the original notebook):
#   1. Households now RECEIVE wages every week (paid by their employer firm).
#   2. Loans carry an outstanding balance; floating-rate loans are re-priced
#      from (OPR + spread) every week, so an OPR shock changes instalments.
#   3. Loan take-up and loan size depend on income class.
#   4. Income class is assigned from DOSM HIES 2022 thresholds, not by index.
#   5. Default = absorbing event when liquid wealth falls below a credit limit;
#      DSR (debt payments / income, excl. living costs) tracked separately.
#   6. Scenarios are run over many seeds with common random numbers.
# =============================================================================

WEEKS_PER_MONTH = 52 / 12

# DOSM HIES 2022 (gross monthly household income, RM)
DOSM_2022 = dict(
    median=6338, mean=8479,
    b40_upper=5249, t20_lower=11820,
    class_mean={"B40": 3401, "M40": 7971, "T20": 19752},
    income_share={"B40": 0.161, "M40": 0.376, "T20": 0.463},
)

# Loan products: (name, floating?, spread over OPR or fixed rate, term range in years)
LOAN_TYPES = ["mortgage", "hire_purchase", "personal"]

DEFAULT_PARAMS = dict(
    # --- Income distribution (lognormal fitted to DOSM median & mean) ---
    income_mu=np.log(DOSM_2022["median"]),
    income_sigma=np.sqrt(2 * np.log(DOSM_2022["mean"] / DOSM_2022["median"])),

    # --- Saving propensity lambda by class (uniform ranges; calibrate via SMM) ---
    lambda_range={"B40": (0.01, 0.10), "M40": (0.15, 0.40), "T20": (0.50, 0.85)},
    lambda_firm=0.95,             # firm saving propensity (0 recovers PDF Eq.1-2 exactly)

    # --- Living costs (share of income at t=0, then indexed to CPI) ---
    essential_share={"B40": 0.75, "M40": 0.62, "T20": 0.45},
    fuel_share={"B40": 0.10, "M40": 0.08, "T20": 0.05},   # fuel's share of essential spend

    # --- Initial liquid buffer, in months of income ---
    buffer_months={"B40": (0.0, 2.0), "M40": (1.0, 4.0), "T20": (3.0, 12.0)},

    # --- Loans: take-up probability by class ---
    loan_prob={
        "mortgage":      {"B40": 0.20, "M40": 0.40, "T20": 0.55},
        "hire_purchase": {"B40": 0.35, "M40": 0.45, "T20": 0.45},
        "personal":      {"B40": 0.30, "M40": 0.20, "T20": 0.10},
    },
    # principal as a multiple of ANNUAL income (uniform range)
    loan_size={"mortgage": (2.5, 4.5), "hire_purchase": (0.4, 1.0), "personal": (0.2, 0.6)},
    # remaining term in years (uniform range)
    loan_term={"mortgage": (5, 30), "hire_purchase": (2, 9), "personal": (2, 10)},
    # pricing: floating loans = OPR + spread; HP is fixed-rate in Malaysia
    loan_floating={"mortgage": True, "hire_purchase": False, "personal": True},
    loan_spread={"mortgage": 1.50, "hire_purchase": 3.00, "personal": 5.00},   # % p.a.

    # --- Unsecured overdraft / credit-card debt when liquid wealth < 0 ---
    card_rate=15.0,               # % p.a., not OPR-linked
    credit_limit_months=2.0,      # default if liquid wealth < -limit * monthly income

    # --- Firms and the open bath ---
    firm_capital=(50_000, 10_000),
    external_demand_share=0.15,   # exports/government demand, as a share of the wage bill
    bank_recycle_share=1.0,       # share of loan repayments re-lent to firms
    firm_payout_rate=0.05,        # weekly share of firm wealth above starting capital paid out
                                  # to the bath (taxes, dividends, imports)

    dsr_threshold=0.60,
)


def scenario_path(name, weeks):
    """Return weekly paths: opr (% p.a.), cpi (annual %), wage growth (annual %), fuel multiplier."""
    t = np.arange(weeks)
    opr = np.full(weeks, 2.75)
    cpi = np.full(weeks, 2.0)
    wage_g = np.full(weeks, 3.0)
    fuel = np.ones(weeks)

    if name in ("opr_hike", "combined"):
        for wk in (13, 26, 39, 52):          # four 25bp hikes -> 3.75%
            opr[t >= wk] += 0.25
    if name in ("subsidy_removal", "combined"):
        fuel[t >= 26] = 2.60 / 1.99           # unsubsidised / BUDI95 RON95 price
        cpi[(t >= 26) & (t < 78)] += 1.0      # one-year pass-through to CPI
    if name in ("inflation_spike", "combined"):
        cpi[(t >= 26) & (t < 130)] = np.maximum(cpi[(t >= 26) & (t < 130)], 6.0)
    return dict(opr=opr, cpi=cpi, wage_g=wage_g, fuel=fuel)


SCENARIOS = ["baseline", "opr_hike", "subsidy_removal", "inflation_spike", "combined"]


def annuity_payment(balance, annual_rate_pct, n_weeks):
    r = annual_rate_pct / 100 / 52
    n = np.maximum(n_weeks, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        pay = np.where(r > 0, balance * r / (1 - (1 + r) ** (-n)), balance / n)
    return np.where(n_weeks > 0, pay, 0.0)


class KineticDebtModel:
    def __init__(self, N_h=120_000, N_f=30_000, params=None, seed=0):
        self.N_h, self.N_f = N_h, N_f
        self.p = {**DEFAULT_PARAMS, **(params or {})}
        self.rng = np.random.default_rng(seed)
        self._init_households()
        self._init_loans()
        self._init_firms()

    # ------------------------------------------------------------------ setup
    def _init_households(self):
        p, rng, N = self.p, self.rng, self.N_h
        self.income0 = rng.lognormal(p["income_mu"], p["income_sigma"], N)   # monthly RM
        self.income0 = np.clip(self.income0, 500, None)
        self.cls = np.where(self.income0 <= DOSM_2022["b40_upper"], 0,
                            np.where(self.income0 < DOSM_2022["t20_lower"], 1, 2))
        self.class_names = np.array(["B40", "M40", "T20"])

        self.lam = np.empty(N)
        self.ess_share = np.empty(N)
        self.fuel_share = np.empty(N)
        buffer = np.empty(N)
        for k, c in enumerate(self.class_names):
            m = self.cls == k
            lo, hi = p["lambda_range"][c]
            self.lam[m] = rng.uniform(lo, hi, m.sum())
            self.ess_share[m] = p["essential_share"][c]
            self.fuel_share[m] = p["fuel_share"][c]
            lo, hi = p["buffer_months"][c]
            buffer[m] = rng.uniform(lo, hi, m.sum())
        self.w = buffer * self.income0                       # liquid wealth, RM
        self.ess0_week = self.ess_share * self.income0 / WEEKS_PER_MONTH
        self.employer = rng.integers(0, self.N_f, N)
        self.defaulted = np.zeros(N, dtype=bool)
        self.default_week = np.full(N, -1)

    def _init_loans(self):
        p, rng, N = self.p, self.rng, self.N_h
        L = len(LOAN_TYPES)
        self.bal = np.zeros((N, L))
        self.n_left = np.zeros((N, L), dtype=int)
        self.fixed_pay = np.zeros((N, L))
        self.floating = np.array([p["loan_floating"][t] for t in LOAN_TYPES])
        self.spread = np.array([p["loan_spread"][t] for t in LOAN_TYPES])
        annual_income = self.income0 * 12
        for j, t in enumerate(LOAN_TYPES):
            prob = np.array([p["loan_prob"][t][c] for c in self.class_names])[self.cls]
            has = rng.random(N) < prob
            lo, hi = p["loan_size"][t]
            self.bal[has, j] = annual_income[has] * rng.uniform(lo, hi, has.sum())
            lo, hi = p["loan_term"][t]
            self.n_left[has, j] = (rng.uniform(lo, hi, has.sum()) * 52).astype(int)
            if not self.floating[j]:   # fixed-rate: payment set at origination (OPR 2.75%)
                rate = self.spread[j]
                self.fixed_pay[has, j] = annuity_payment(self.bal[has, j], rate, self.n_left[has, j])

    def _init_firms(self):
        mu, sd = self.p["firm_capital"]
        self.wf = self.rng.normal(mu, sd, self.N_f)
        self.wf0 = self.wf.copy()
        # firm size = its wage bill; customers' spending reaches firms in proportion to size
        bill = np.bincount(self.employer, weights=self.income0, minlength=self.N_f)
        size = bill + 0.1 * bill.mean()
        self.firm_weight = size / size.sum()

    # --------------------------------------------------------------- dynamics
    def step(self, t, opr, price_level, wage_level, fuel_mult):
        p, rng = self.p, self.rng
        N, Nf = self.N_h, self.N_f

        # 1. Wages: employer firm -> household
        wage = self.income0 * wage_level / WEEKS_PER_MONTH
        self.w += wage
        np.subtract.at(self.wf, self.employer, wage)

        # 2. Open bath: external demand (exports / government) flows into firms
        self.wf += p["external_demand_share"] * wage.sum() * self.firm_weight

        # 3. Essential spending (CPI-indexed, fuel component hit by subsidy removal)
        ess = self.ess0_week * price_level * (1 + self.fuel_share * (fuel_mult - 1))
        self.w -= ess
        self.wf += ess.sum() * self.firm_weight

        # 4. Loan servicing (balance-based; floating loans re-priced at OPR + spread)
        rate = np.where(self.floating, opr + self.spread, self.spread)          # % p.a. per type
        float_pay = annuity_payment(self.bal, rate[None, :], self.n_left)
        pay = np.where(self.floating[None, :], float_pay, self.fixed_pay)
        pay = np.where(self.n_left > 0, np.minimum(pay, self.bal * (1 + rate / 100 / 52)), 0.0)
        interest = self.bal * rate[None, :] / 100 / 52
        self.bal = np.maximum(self.bal + interest - pay, 0.0)
        self.n_left = np.where(self.n_left > 0, self.n_left - 1, 0)
        total_pay = pay.sum(axis=1)
        self.w -= total_pay

        # 5. Unsecured debt (Eq.3): interest on negative liquid wealth at card rate
        neg = self.w < 0
        card_interest = np.zeros(N)
        card_interest[neg] = -self.w[neg] * p["card_rate"] / 100 / 52
        self.w -= card_interest
        bank_inflow = total_pay.sum() + card_interest.sum()
        self.wf += p["bank_recycle_share"] * bank_inflow * self.firm_weight

        # 6. Kinetic exchange (Chakraborti-Chakrabarti, bipartite, 1-to-1 matching)
        #    Only households with positive liquid wealth trade (you cannot spend debt).
        hh = rng.choice(N, Nf, replace=False)
        wh = np.maximum(self.w[hh], 0.0)
        wf = np.maximum(self.wf, 0.0)
        lam_h, lam_f = self.lam[hh], p["lambda_firm"]
        eps = rng.random(Nf)
        pool = (1 - lam_h) * wh + (1 - lam_f) * wf
        new_wh = lam_h * wh + eps * pool
        new_wf = lam_f * wf + (1 - eps) * pool
        self.w[hh] += new_wh - wh
        self.wf += new_wf - wf

        # 6b. Firms pay out aggregate profit (wealth above starting capital) to the bath
        agg_excess = self.wf.sum() - self.wf0.sum()
        if agg_excess > 0:
            excess = np.maximum(self.wf - self.wf0, 0.0)
            self.wf -= p["firm_payout_rate"] * agg_excess * excess / excess.sum()

        # 7. Distress metrics
        monthly_income = self.income0 * wage_level
        monthly_debt_service = (total_pay + card_interest) * WEEKS_PER_MONTH
        dsr = monthly_debt_service / monthly_income
        limit = -p["credit_limit_months"] * monthly_income
        new_def = (~self.defaulted) & (self.w < limit)
        self.defaulted |= new_def
        self.default_week[new_def] = t
        return dsr

    def run(self, scenario="baseline", weeks=260):
        path = scenario_path(scenario, weeks)
        price, wage_level = 1.0, 1.0
        out = {k: np.zeros((weeks, 3)) for k in ("dsr60", "negative", "pd_cum")}
        out["firm_wealth"] = np.zeros(weeks)
        out["firm_distress"] = np.zeros(weeks)
        for t in range(weeks):
            price *= (1 + path["cpi"][t] / 100) ** (1 / 52)
            wage_level *= (1 + path["wage_g"][t] / 100) ** (1 / 52)
            dsr = self.step(t, path["opr"][t], price, wage_level, path["fuel"][t])
            for k in range(3):
                m = self.cls == k
                out["dsr60"][t, k] = np.mean(dsr[m] > self.p["dsr_threshold"])
                out["negative"][t, k] = np.mean(self.w[m] < 0)
                out["pd_cum"][t, k] = np.mean(self.defaulted[m])
            out["firm_wealth"][t] = self.wf.mean()
            out["firm_distress"][t] = np.mean(self.wf < 0)
        out["income0"] = self.income0
        out["cls"] = self.cls
        out["defaulted"] = self.defaulted.copy()
        out["path"] = path
        return out


def monte_carlo(scenarios=SCENARIOS, n_seeds=10, weeks=260, N_h=30_000, N_f=7_500,
                params=None, verbose=True):
    """Run every scenario over the same seeds (common random numbers)."""
    results = {s: [] for s in scenarios}
    for seed in range(n_seeds):
        for s in scenarios:
            m = KineticDebtModel(N_h, N_f, params, seed=seed)
            results[s].append(m.run(s, weeks))
        if verbose:
            print(f"seed {seed + 1}/{n_seeds} done")
    return results

In [ ]:
# Chart style (thin marks, recessive grid, fixed colour order)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6, "lines.linewidth": 2, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
})
SCEN_COLOR = dict(zip(SCENARIOS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
SCEN_LABEL = {"baseline": "Baseline", "opr_hike": "OPR +100bp", "subsidy_removal": "RON95 subsidy removed",
              "inflation_spike": "Inflation spike (6%)", "combined": "All three combined"}
CLASSES = ["B40", "M40", "T20"]

## 2. Calibration check against DOSM HIES 2022
Household income is drawn from a lognormal fitted to the DOSM 2022 median (RM6,338) and mean (RM8,479); classes are then assigned with DOSM's thresholds (B40 ≤ RM5,249, T20 ≥ RM11,820).

In [ ]:
m0 = KineticDebtModel(N_h=120_000, N_f=30_000, seed=0)
rows = []
for k, c in enumerate(CLASSES):
    sel = m0.cls == k
    rows.append({
        "Class": c,
        "Share of households (sim)": sel.mean(),
        "Mean income RM (sim)": m0.income0[sel].mean(),
        "Mean income RM (DOSM 2022)": DOSM_2022["class_mean"][c],
        "Share of total income (sim)": m0.income0[sel].sum() / m0.income0.sum(),
        "Share of total income (DOSM 2022)": DOSM_2022["income_share"][c],
        "Households with a loan (sim)": (m0.bal[sel].sum(1) > 0).mean(),
    })
calib = pd.DataFrame(rows).set_index("Class")
display(calib.style.format({c: "{:.1%}" for c in calib.columns if "Share" in c or "loan" in c} |
                           {c: "RM{:,.0f}" for c in calib.columns if "RM" in c}))

fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, 40_000, 81)
for k, c in enumerate(CLASSES):
    ax.hist(m0.income0[m0.cls == k], bins=bins, color=SCEN_COLOR[SCENARIOS[k]], label=c,
            edgecolor="white", linewidth=0.5)
for x in (DOSM_2022["b40_upper"], DOSM_2022["t20_lower"]):
    ax.axvline(x, color="#6b6a63", lw=1, ls="--")
ax.set(title="Simulated monthly household income by class", xlabel="Gross monthly income (RM)",
       ylabel="Households")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Shock scenarios
OPR starts at 2.75% (BNM rate since July 2025). The fuel multiplier 2.60 / 1.99 is the unsubsidised RON95 price over the BUDI95 price (Oct 2025); for a harsher stress test use 3.87 / 1.99 (late March 2026). Edit `scenario_path` in the engine cell to change any scenario.

In [ ]:
WEEKS = 260   # 5 years: 2025-2030, as in the research note
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
for s in SCENARIOS:
    p = scenario_path(s, WEEKS)
    yrs = np.arange(WEEKS) / 52
    for ax, key in zip(axes, ["opr", "cpi", "fuel"]):
        ax.plot(yrs, p[key], color=SCEN_COLOR[s], label=SCEN_LABEL[s])
axes[0].set(title="OPR (% p.a.)", xlabel="Years")
axes[1].set(title="CPI inflation (% p.a.)", xlabel="Years")
axes[2].set(title="Fuel price multiplier", xlabel="Years")
axes[2].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(); plt.show()

## 4. Monte Carlo runs
Every scenario is run on the **same seeds** (common random numbers), so differences between scenarios come from the shock, not from luck.

In [ ]:
FULL_SIZE = False   # True = 120,000 households / 30,000 firms (about 4 s per run)
N_SEEDS = 20

N_h, N_f = (120_000, 30_000) if FULL_SIZE else (30_000, 7_500)
results = monte_carlo(SCENARIOS, n_seeds=N_SEEDS, weeks=WEEKS, N_h=N_h, N_f=N_f, verbose=False)
print(f"Done: {len(SCENARIOS)} scenarios x {N_SEEDS} seeds, {N_h:,} households, {WEEKS} weeks")

## 5. Results
### 5.1 Probability of default after 5 years
Default = liquid wealth falls below −2 months of income (the household has exhausted its credit line). The table shows the mean across seeds, the 5–95% range, and the change versus baseline (paired by seed).

In [ ]:
def pd_final(s):
    return np.array([r["pd_cum"][-1] for r in results[s]])   # shape (seeds, 3)

base = pd_final("baseline")
rows = []
for s in SCENARIOS:
    a = pd_final(s); d = a - base
    for k, c in enumerate(CLASSES):
        rows.append({"Scenario": SCEN_LABEL[s], "Class": c,
                     "PD (mean)": a[:, k].mean(),
                     "PD 5%": np.percentile(a[:, k], 5), "PD 95%": np.percentile(a[:, k], 95),
                     "Change vs baseline (pp)": 100 * d[:, k].mean(),
                     "Relative change": d[:, k].mean() / base[:, k].mean()})
summary = pd.DataFrame(rows)
display(summary.pivot(index="Scenario", columns="Class", values="PD (mean)")
        .loc[[SCEN_LABEL[s] for s in SCENARIOS]].style.format("{:.2%}")
        .set_caption("Cumulative probability of default after 5 years"))
display(summary.style.format({"PD (mean)": "{:.2%}", "PD 5%": "{:.2%}", "PD 95%": "{:.2%}",
                              "Change vs baseline (pp)": "{:+.2f}", "Relative change": "{:+.0%}"}).hide(axis="index"))

### 5.2 How default risk builds up over time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
yrs = np.arange(WEEKS) / 52
for k, (ax, c) in enumerate(zip(axes, CLASSES)):
    for s in SCENARIOS:
        paths = np.array([r["pd_cum"][:, k] for r in results[s]]) * 100
        lo, mid, hi = np.percentile(paths, [5, 50, 95], axis=0)
        ax.fill_between(yrs, lo, hi, color=SCEN_COLOR[s], alpha=0.15, lw=0)
        ax.plot(yrs, mid, color=SCEN_COLOR[s], label=SCEN_LABEL[s])
    ax.set(title=c, xlabel="Years")
axes[0].set_ylabel("Cumulative default (% of class)")
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.suptitle("Cumulative probability of default by income class (median, 5–95% band)", y=1.02, fontweight="bold")
plt.tight_layout(); plt.show()

### 5.3 Default probability at year 5, by class and scenario

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
width = 0.16
x = np.arange(len(CLASSES))
for i, s in enumerate(SCENARIOS):
    a = pd_final(s) * 100
    mean = a.mean(0); lo = mean - np.percentile(a, 5, axis=0); hi = np.percentile(a, 95, axis=0) - mean
    ax.bar(x + (i - 2) * width, mean, width * 0.9, color=SCEN_COLOR[s], label=SCEN_LABEL[s],
           yerr=[lo, hi], error_kw=dict(ecolor="#6b6a63", lw=1, capsize=2))
ax.set_xticks(x, CLASSES)
ax.set(ylabel="Default probability (%)", title="Probability of default after 5 years")
ax.grid(axis="x", visible=False)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout(); plt.show()

### 5.4 Default risk across the income distribution
Income deciles (10 equal groups of households). This is an *output* of the model: loans, buffers and living costs differ by class, but nothing forces the shape of this curve.

> **Read with care:** sharp kinks near deciles 4 and 8 sit at the B40/M40 and M40/T20 thresholds. They come from placeholder parameters (loan take-up, buffers, living costs) that jump at class boundaries. Once you have data, make these parameters vary smoothly with income and the kinks should disappear.

In [ ]:
def pd_by_decile(s):
    out = []
    for r in results[s]:
        edges = np.percentile(r["income0"], np.arange(0, 101, 10))
        d = np.clip(np.digitize(r["income0"], edges[1:-1]), 0, 9)
        out.append([r["defaulted"][d == i].mean() for i in range(10)])
    return np.array(out) * 100

fig, ax = plt.subplots(figsize=(9, 3.8))
for s in ["baseline", "combined"]:
    a = pd_by_decile(s)
    ax.plot(np.arange(1, 11), a.mean(0), marker="o", ms=6, color=SCEN_COLOR[s], label=SCEN_LABEL[s])
    ax.fill_between(np.arange(1, 11), np.percentile(a, 5, 0), np.percentile(a, 95, 0), color=SCEN_COLOR[s], alpha=0.15, lw=0)
ax.set(xticks=range(1, 11), xlabel="Income decile (1 = poorest 10%)", ylabel="Default probability (%)",
       title="5-year default probability by income decile")
ax.legend(); plt.tight_layout(); plt.show()

### 5.5 Debt-service stress and firm health
DSR here = monthly loan instalments + card interest, divided by gross monthly income (living costs excluded, as in BNM's definition). Firms are tracked to check the money flows stay balanced.

> **Known limitations:** (1) no new loans are originated during the run, so the DSR share falls as existing loans amortise and wages grow; (2) the first ~6 months are a warm-up period while firm wealth settles (consider discarding them, or starting the clock after a burn-in).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for s in ["baseline", "combined"]:
    dsr = np.array([r["dsr60"] for r in results[s]]).mean(0) * 100
    for k, c in enumerate(CLASSES):
        axes[0].plot(yrs, dsr[:, k], color=SCEN_COLOR[SCENARIOS[k]],
                     ls="-" if s == "baseline" else "--", label=f"{c} – {SCEN_LABEL[s]}")
    fw = np.array([r["firm_wealth"] for r in results[s]]).mean(0)
    fd = np.array([r["firm_distress"] for r in results[s]]).mean(0) * 100
    axes[1].plot(yrs, fw, color=SCEN_COLOR[s], label=SCEN_LABEL[s])
    axes[2].plot(yrs, fd, color=SCEN_COLOR[s], label=SCEN_LABEL[s])
axes[0].set(title="Households with DSR > 60% (%)", xlabel="Years")
axes[0].legend(fontsize=7.5)
axes[1].set(title="Mean firm wealth (RM)", xlabel="Years")
axes[2].set(title="Firms with negative wealth (%)", xlabel="Years")
axes[2].legend()
plt.tight_layout(); plt.show()

## 6. Validation helpers
**Two-sample K-S test.** The null hypothesis is that the two samples come from the same distribution, so **p > 0.05 means no evidence of a mismatch** (the research note states this the other way round). With 100,000+ agents the test detects even tiny differences, so also report the **D statistic** (the maximum gap between the two CDFs) and compare shares by decile.

Upload real data (for example a CSV of household incomes or wealth) with the file browser on the left or `files.upload()`, then call `ks_compare`.

In [ ]:
from scipy import stats

def ks_compare(simulated, empirical, label="income"):
    D, p = stats.ks_2samp(simulated, empirical)
    verdict = "no evidence of mismatch" if p > 0.05 else "distributions differ"
    print(f"K-S on {label}: D = {D:.4f}, p = {p:.4g}  ->  {verdict}")
    q = np.arange(10, 100, 10)
    return pd.DataFrame({"Simulated": np.percentile(simulated, q), "Empirical": np.percentile(empirical, q)},
                        index=[f"P{v}" for v in q]).style.format("{:,.0f}")

def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())

print(f"Simulated income Gini: {gini(m0.income0):.3f}  (DOSM 2022 reported 0.404)")

# Example once you have real data:
# from google.colab import files; up = files.upload()
# emp = pd.read_csv(next(iter(up)))["income"].to_numpy()
# ks_compare(m0.income0, emp)

## 7. Parameters to replace with real data, and next steps
| Parameter (in `DEFAULT_PARAMS`) | Current value | Suggested source |
|---|---|---|
| `lambda_range` (saving propensity) | B40 0.01–0.10, M40 0.15–0.40, T20 0.50–0.85 | Calibrate with SMM against DOSM HES savings / BNM household wealth moments |
| `loan_prob`, `loan_size`, `loan_term` | illustrative | BNM Financial Stability Review (household debt by income group), CCRIS statistics |
| `loan_spread` | mortgage 1.5%, HP 3.0% fixed, personal 5.0% | BNM average lending rates by product |
| `essential_share`, `fuel_share` | B40 75% / 10%, M40 62% / 8%, T20 45% / 5% | DOSM Household Expenditure Survey 2022 by income group |
| `buffer_months` | B40 0–2, M40 1–4, T20 3–12 | BNM FSR liquid-asset buffers |
| `card_rate`, `credit_limit_months` | 15%, 2 months | BNM credit card regulations, AKPK |
| `external_demand_share`, `firm_payout_rate` | 0.15, 0.05 | Tune so firm wealth stays stable in the baseline |

**Next steps:** (1) replace placeholders with data; (2) add an SMM loop that tunes `lambda_range` to empirical moments for 2011–2020; (3) validate the 2021–2024 forward run with `ks_compare` and decile shares; (4) optional extensions: unemployment shocks when firms go insolvent, AKPK restructuring for defaulted households, regional (urban/rural) thresholds.